In [1]:
import pandas as pd
import json
import datetime

In [2]:
cities = pd.read_csv('../dataset/cities.csv', sep=',')
airlines = pd.read_csv('../dataset/airlines.csv', sep=',')
flights = pd.read_csv('../dataset/flights.csv', sep=',')
logs = pd.read_csv('../dataset/logs.csv', sep=',')
airports = pd.read_csv('../dataset/airports.csv', sep=',')

In [17]:
print("cities shape:", cities.shape)
print("airlines shape:", airlines.shape)
print("flights shape:", flights.shape)
print("logs shape:", logs.shape)
print("airports shape:", airports.shape)

cities shape: (316, 9)
airlines shape: (19, 2)
flights shape: (5262836, 20)
logs shape: (5765, 10)
airports shape: (342, 7)


In [18]:
print(cities.columns)
print(airlines.columns)
print(flights.columns)
print(logs.columns)
print(airports.columns)

Index(['city', 'city_ascii', 'state_id', 'state_name', 'lat', 'lng',
       'population', 'density', 'timezone'],
      dtype='str')
Index(['IATA_CODE', 'AIRLINE'], dtype='str')
Index(['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER',
       'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE',
       'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'SCHEDULED_TIME', 'ELAPSED_TIME',
       'AIR_TIME', 'DISTANCE', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME',
       'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED'],
      dtype='str')
Index(['flight', 'lat', 'lon', 'alt', 'gspeed', 'vspeed', 'timestamp',
       'orig_iata', 'dest_iata', 'eta'],
      dtype='str')
Index(['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY', 'LATITUDE',
       'LONGITUDE'],
      dtype='str')


In [20]:
print(airports['LATITUDE'].dtype)
print(airports['LONGITUDE'].dtype)

float64
float64


In [ ]:
airports.query("IATA_CODE == 'AKL'")

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
13,AKL,Auckland Airport,Auckland,NaN,New Zealand,-3.70082,17.47916


In [17]:
airports.isna().sum()

IATA_CODE    0
AIRPORT      0
CITY         0
STATE        1
COUNTRY      0
LATITUDE     3
LONGITUDE    3
dtype: int64

In [4]:
flights.isna().sum()

YEAR                         0
MONTH                        0
DAY                          0
DAY_OF_WEEK                  0
AIRLINE                      0
FLIGHT_NUMBER                0
ORIGIN_AIRPORT               0
DESTINATION_AIRPORT          0
SCHEDULED_DEPARTURE          0
DEPARTURE_TIME         1597627
DEPARTURE_DELAY        1597627
SCHEDULED_TIME               0
ELAPSED_TIME             38997
AIR_TIME                 38997
DISTANCE                     0
SCHEDULED_ARRIVAL            0
ARRIVAL_TIME           1597789
ARRIVAL_DELAY          1606083
DIVERTED                     0
CANCELLED                    0
dtype: int64

In [6]:
flights.shape

(5262834, 20)

# File JSON airports

In [28]:
import pandas as pd
import json
import datetime

FILE_AIRLINES = '../dataset/airlines.csv'
FILE_AIRPORTS = '../dataset/airports.csv'
FILE_FLIGHTS = '../dataset/flights.csv'

OUTPUT_AIRPORTS = '../dataset/json/airports.json'
OUTPUT_FLIGHTS = '../dataset/json/flights.json'

print("1. Caricamento Lookup Tables (Airlines & Airports)...")

# 1. Carichiamo AIRLINES in un dizionario
airlines_df = pd.read_csv(FILE_AIRLINES, dtype=str)
airlines_map = dict(zip(airlines_df['IATA_CODE'], airlines_df['AIRLINE']))

# 2. Carichiamo AIRPORTS
airports_df = pd.read_csv(FILE_AIRPORTS, dtype=str)
airports_map = {}
airports_collection_data = []

counter = 0

for _, row in airports_df.iterrows():
    # if counter == 5:
    #     break
    try:
        lat = float(row['latitude'])
        lon = float(row['longitude'])
        
        # Oggetto GeoJSON standard
        geo_location = {
            "type": "Point",
            "coordinates": [lon, lat] if pd.notnull(lat) and pd.notnull(lon) else None
        }
        
        airport_obj = {
            "_id": row['iata_code'],
            "name": row['airport'],
            "city": row['city'],
            "state": row['state'] if pd.notnull(row['state']) else row['country'],
            "country": row['country'],
            "location": geo_location
        }
        
        airports_map[row['iata_code']] = airport_obj
        airports_collection_data.append(airport_obj)
        
    except (ValueError, TypeError):
        print(f"Warning: Coordinate non valida per aeroporto {row['iata_code']} - Skipping")
        continue

    counter += 1

# --- MODIFICA QUI ---
# Scriviamo il file JSON come un unico array valido
# with open(OUTPUT_AIRPORTS, 'w', encoding='utf-8') as f:
#     json.dump(airports_collection_data, f, ensure_ascii=False)

print(f"   -> Creato {OUTPUT_AIRPORTS} con {len(airports_collection_data)} aeroporti.")

1. Caricamento Lookup Tables (Airlines & Airports)...
   -> Creato ../dataset/json/airports.json con 342 aeroporti.


# Flights JSON

In [29]:
print("2. Elaborazione Voli e Denormalizzazione (Questo richiederà tempo)...")

# 3. Processiamo i VOLI in streaming (chunksize) per non intasare la RAM
count = 0
is_first_record = True # Flag per gestire la virgola

with open(OUTPUT_FLIGHTS, 'w') as f_out:
    f_out.write('[') # 1. Apriamo l'array JSON all'inizio del file
    
    # Leggiamo il CSV a blocchi
    for chunk in pd.read_csv(FILE_FLIGHTS, dtype=str, chunksize=100000):
        for _, row in chunk.iterrows():
            
            # Recuperiamo i dati dai lookup
            airline_code = row['AIRLINE']
            origin_code = row['ORIGIN_AIRPORT']
            dest_code = row['DESTINATION_AIRPORT']

            # Se l'aeroporto non esiste nel lookup (es. codici vecchi), saltiamo o gestiamo
            origin_data = airports_map.get(origin_code)
            dest_data = airports_map.get(dest_code)
            
            if not origin_data or not dest_data:
                # print(f"   Attenzione: Aeroporto non trovato per volo {row['FLIGHT_NUMBER']} - ORIGIN: {origin_code}, DEST: {dest_code}. Skipping...")
                continue

            # Costruzione della Data
            try:
                flight_date = datetime.datetime(
                    int(row['YEAR']), int(row['MONTH']), int(row['DAY'])
                ).isoformat()
            except:
                flight_date = None

            # --- COSTRUZIONE DEL DOCUMENTO DENORMALIZZATO ---
            doc = {
                # Dati Volo
                "flight_info": {
                    "flight_key": row['FLIGHT_KEY'],
                    "airline": {
                        "iata": airline_code,
                        "name": airlines_map.get(airline_code, "Unknown")
                    },
                    "schedule": {
                        "date": flight_date,
                        "departure": row['SCHEDULED_DEPARTURE'],
                        "arrival": row['SCHEDULED_ARRIVAL'],
                        "duration_minutes": float(row['SCHEDULED_TIME']) if pd.notnull(row['SCHEDULED_TIME']) else None
                    },
                },
                
                # Dati Rotta (EMBEDDED AIRPORTS)
                "route": {
                    "origin": {
                        "iata": origin_code,
                        "airport_name": origin_data['name'],
                        "city": origin_data['city'],
                        "state": origin_data['state'],
                        "country": origin_data['country'],
                        "location": origin_data['location']
                    },
                    "destination": {
                        "iata": dest_code,
                        "airport_name": dest_data['name'],
                        "city": dest_data['city'],
                        "state": dest_data['state'],
                        "country": dest_data['country'],
                        "location": dest_data['location']
                    },
                    "distance_km": float(row['DISTANCE']) if pd.notnull(row['DISTANCE']) else None
                },
                
                # Dati Performance
                "stats": {
                    "tot_delay_minutes": float(row['DEPARTURE_DELAY']) + float(row['ARRIVAL_DELAY']) if pd.notnull(row['DEPARTURE_DELAY']) and pd.notnull(row['ARRIVAL_DELAY']) else None,
                    "is_cancelled": int(row['CANCELLED']) if pd.notnull(row['CANCELLED']) else None,
                    "is_diverted": int(row['DIVERTED']) if pd.notnull(row['DIVERTED']) else None,
                    "air_time_minutes": float(row['AIR_TIME']) if pd.notnull(row['AIR_TIME']) else None
                },
                
                "flight_log": None 
            }
            
            # --- MODIFICA PER SCRIVERE JSON VALIDO ---
            if not is_first_record:
                f_out.write(',\n') # 2. Aggiungi virgola e a capo PRIMA del nuovo oggetto (se non è il primo)
            else:
                is_first_record = False # Dopo il primo giro, il flag diventa False

            f_out.write(json.dumps(doc)) # Scriviamo l'oggetto
            
            count += 1
        # Aggiornamento progresso fuori dal loop riga, ma dentro il chunk per non intasare la console
        print(f"   ...processati {count} voli", end='\r')
            
    f_out.write(']') # 3. Chiudiamo l'array JSON alla fine del file

print(f"\n   -> Completato! Creato {OUTPUT_FLIGHTS} con {count} documenti.")

2. Elaborazione Voli e Denormalizzazione (Questo richiederà tempo)...
   ...processati 4772506 voli
   -> Completato! Creato ../dataset/json/flights.json con 4772506 documenti.


In [76]:
airports.head()

,iata_code,airport,city,state,country,latitude,longitude,city_state
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040,allentown_pa_usa
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190,abilene_tx_usa
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919,albuquerque_nm_usa
3,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183,aberdeen_sd_usa
4,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447,albany_ga_usa


In [89]:
routes = flights.groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']).agg(
    flight_count=('ORIGIN_AIRPORT', 'size'),
    avg_scheduled_flight_duration_minutes=('SCHEDULED_TIME', 'mean')
).reset_index()

In [92]:
logs.head()

,flight,lat,lon,alt,gspeed,vspeed,timestamp,orig_iata,dest_iata,eta
0,AA1349,32.81462,-95.80302,40000,346,0,2026-02-25T15:59:57Z,JFK,AUS,2026-02-25T16:34:08Z
1,DL2301,31.69109,-85.77339,39000,502,0,2026-02-25T15:59:58Z,MSP,TPA,2026-02-25T16:44:48Z
2,DL2582,35.93753,-115.38531,39000,436,0,2026-02-25T15:59:59Z,BUR,SLC,2026-02-25T16:59:44Z
3,DL8851,37.86596,-100.55422,40000,368,0,2026-02-25T15:59:59Z,ATL,DEN,2026-02-25T16:41:04Z
4,UA1384,38.17987,-101.32871,38025,386,0,2026-02-25T15:59:58Z,ICT,DEN,2026-02-25T16:40:32Z


In [95]:
flights.query("ORIGIN_AIRPORT == 'JFK' and DESTINATION_AIRPORT == 'AUS' and YEAR == 2026 and MONTH == 2 and DAY == 25")

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED
3665050,2026,2,25,3,AA,1349,JFK,AUS,726,726.0,0.0,254.0,NaN,NaN,2554.24,1040.0,NaN,NaN,0,0
3665056,2026,2,25,3,AA,1349,JFK,AUS,726,726.0,0.0,254.0,NaN,NaN,2554.24,1040.0,NaN,NaN,0,0


In [103]:
flights.shape

(5262834, 20)

In [98]:
flights.drop_duplicates(inplace=True)
flights.shape

(5262528, 20)

In [104]:
unique_flights = flights.drop_duplicates()
unique_flights.shape

(5262528, 20)

In [106]:
flights.columns

Index(['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER',
       'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE',
       'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'SCHEDULED_TIME', 'ELAPSED_TIME',
       'AIR_TIME', 'DISTANCE', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME',
       'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED'],
      dtype='str')

In [109]:
flights.groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'AIRLINE', 'YEAR', 'MONTH', 'DAY', 'SCHEDULED_DEPARTURE']).size().reset_index(name='flight_count').sort_values(by='flight_count', ascending=False).head(10)


,ORIGIN_AIRPORT,DESTINATION_AIRPORT,AIRLINE,YEAR,MONTH,DAY,SCHEDULED_DEPARTURE,flight_count
45921,ANC,ADQ,AS,2026,1,13,1530,8
1448628,DFW,SNA,AA,2025,10,27,855,8
237573,ATL,MCO,DL,2025,10,12,655,8
2554407,LAX,SEA,AS,2026,1,17,600,8
518393,BOS,ATL,DL,2025,11,19,1600,8
1906251,HOU,DAL,WN,2025,9,27,1300,8
1907268,HOU,DAL,WN,2025,12,1,2000,7
977208,DAL,HOU,WN,2026,1,17,1630,7
4052899,SAN,PDX,AS,2025,10,16,630,7
3980118,ROC,ATL,DL,2026,1,21,600,7


In [110]:
flights.query("ORIGIN_AIRPORT == 'ANC' and DESTINATION_AIRPORT == 'ADQ' and YEAR == 2026 and MONTH == 1 and DAY == 13")

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED
2806004,2026,1,13,2,AS,49,ANC,ADQ,1530,1516.0,-14.0,54.0,57.0,45.0,253.0,1624.0,1613.0,-11.0,0,0
2809078,2026,1,13,2,AS,49,ANC,ADQ,1530,1525.0,-5.0,55.0,54.0,43.0,253.0,1625.0,1619.0,-6.0,0,0
2810699,2026,1,13,2,AS,49,ANC,ADQ,1530,1529.0,-1.0,62.0,64.0,44.0,253.0,1632.0,1633.0,1.0,0,0
2814702,2026,1,13,2,AS,49,ANC,ADQ,1530,1528.0,-2.0,57.0,69.0,43.0,253.0,1627.0,1637.0,10.0,0,0
2815241,2026,1,13,2,AS,49,ANC,ADQ,1530,1526.0,-4.0,57.0,52.0,38.0,253.0,1627.0,1618.0,-9.0,0,0
2816402,2026,1,13,2,AS,49,ANC,ADQ,1530,1542.0,12.0,58.0,68.0,48.0,253.0,1628.0,1650.0,22.0,0,0
2819809,2026,1,13,2,AS,49,ANC,ADQ,1530,1545.0,15.0,56.0,62.0,49.0,253.0,1626.0,1647.0,21.0,0,0
2824732,2026,1,13,2,AS,49,ANC,ADQ,1530,1516.0,-14.0,56.0,66.0,43.0,253.0,1626.0,1622.0,-4.0,0,0


In [120]:
duplicates = flights[flights.duplicated()]

In [121]:
duplicates.shape

(0, 20)

In [113]:
duplicates.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED
46792,2025,8,31,7,WN,1636,HOU,DAL,615,611.0,-4.0,60.0,56.0,42.0,239.0,715.0,707.0,-8.0,0,0
244777,2025,9,9,2,WN,3100,ROC,MCO,1310,1307.0,-3.0,185.0,161.0,142.0,1033.0,1615.0,1548.0,-27.0,0,0
275666,2025,9,11,4,DL,1498,FLL,LGA,700,655.0,-5.0,175.0,181.0,146.0,1076.0,955.0,956.0,1.0,0,0
382092,2025,9,16,2,AA,28,LAX,MIA,1220,1216.0,-4.0,298.0,299.0,279.0,2342.0,2018.0,2015.0,-3.0,0,0
520126,2025,9,23,2,WN,725,MSY,BWI,955,1001.0,6.0,150.0,144.0,131.0,998.0,1325.0,1325.0,0.0,0,0


In [ ]:
flights.shape

(5262528, 20)

In [6]:
dup = flights.duplicated(subset=['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'AIRLINE', 'YEAR', 'MONTH', 'DAY', 'SCHEDULED_DEPARTURE'])

In [7]:
# Mostra le righe duplicate (esclusa la prima occorrenza)
flights[dup].sort_values(by=['YEAR', 'MONTH', 'DAY', 'SCHEDULED_DEPARTURE'])

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED
20287,2025,8,29,5,AA,1441,RNO,DFW,30,48.0,18.0,193.0,183.0,166.0,1345.0,543.0,551.0,8.0,0,0
15152,2025,8,29,5,DL,2379,PDX,MSP,40,35.0,-5.0,193.0,190.0,170.0,1426.0,553.0,545.0,-8.0,0,0
14125,2025,8,29,5,AS,108,ANC,SEA,45,47.0,2.0,201.0,203.0,189.0,1448.0,506.0,510.0,4.0,0,0
12149,2025,8,29,5,NK,214,LAS,DFW,50,49.0,-1.0,154.0,148.0,130.0,1055.0,524.0,517.0,-7.0,0,0
9383,2025,8,29,5,AS,114,ANC,SEA,245,240.0,-5.0,195.0,198.0,182.0,1448.0,700.0,658.0,-2.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5259208,2026,8,23,7,DL,2509,SFO,ATL,2255,NaN,NaN,265.0,252.0,223.0,2139.0,620.0,NaN,NaN,0,0
5260047,2026,8,23,7,DL,2416,ATL,TLH,2305,NaN,NaN,64.0,55.0,35.0,223.0,9.0,NaN,NaN,0,0
5260754,2026,8,23,7,AS,150,ANC,LAX,2350,NaN,NaN,325.0,307.0,289.0,2345.0,615.0,NaN,NaN,0,0
5258038,2026,8,23,7,AA,1522,SFO,ORD,2355,NaN,NaN,244.0,244.0,224.0,1846.0,559.0,NaN,NaN,0,0


In [30]:
logs.head()

,flight,lat,lon,alt,gspeed,vspeed,timestamp,orig_iata,dest_iata,eta
0,AA1349,32.81462,-95.80302,40000,346,0,2026-02-25T15:59:57Z,JFK,AUS,2026-02-25T16:34:08Z
1,DL2301,31.69109,-85.77339,39000,502,0,2026-02-25T15:59:58Z,MSP,TPA,2026-02-25T16:44:48Z
2,DL2582,35.93753,-115.38531,39000,436,0,2026-02-25T15:59:59Z,BUR,SLC,2026-02-25T16:59:44Z
3,DL8851,37.86596,-100.55422,40000,368,0,2026-02-25T15:59:59Z,ATL,DEN,2026-02-25T16:41:04Z
4,UA1384,38.17987,-101.32871,38025,386,0,2026-02-25T15:59:58Z,ICT,DEN,2026-02-25T16:40:32Z


In [32]:
flights.query("ORIGIN_AIRPORT == 'JFK' and DESTINATION_AIRPORT == 'AUS' and YEAR == 2026 and MONTH == 2 and DAY == 25")

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,...,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,FLIGHT_KEY
3258206,2026,2,25,3,AA,1349,JFK,AUS,726,726.0,...,254.0,NaN,NaN,2554.24,1040.0,NaN,NaN,0,0,JFK_AUS_AA_2026_2_25_726


In [ ]:
flights.query("YEAR == 2026 and MONTH > 2")

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,...,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,FLIGHT_KEY
3283942,2026,3,1,7,WN,384,PHX,SDF,1045,NaN,...,205.0,193.0,174.0,1506.0,1710.0,NaN,NaN,0,0,PHX_SDF_WN_2026_3_1_1045
3283943,2026,3,1,7,AA,114,ORD,SNA,705,NaN,...,265.0,258.0,226.0,1726.0,930.0,NaN,NaN,0,0,ORD_SNA_AA_2026_3_1_705
3283944,2026,3,1,7,OO,4988,LAS,LAX,1039,NaN,...,84.0,83.0,47.0,236.0,1203.0,NaN,NaN,0,0,LAS_LAX_OO_2026_3_1_1039
3283945,2026,3,1,7,DL,2134,MSP,PHX,1125,NaN,...,187.0,178.0,157.0,1276.0,1232.0,NaN,NaN,0,0,MSP_PHX_DL_2026_3_1_1125
3283946,2026,3,1,7,B6,1667,BOS,CHS,1034,NaN,...,140.0,130.0,110.0,818.0,1254.0,NaN,NaN,0,0,BOS_CHS_B6_2026_3_1_1034
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4772501,2026,8,23,7,EV,6164,ORD,CID,1754,NaN,...,66.0,66.0,40.0,196.0,1900.0,NaN,NaN,0,0,ORD_CID_EV_2026_8_23_1754
4772502,2026,8,23,7,WN,2410,OAK,SAN,2050,NaN,...,90.0,81.0,68.0,446.0,2220.0,NaN,NaN,0,0,OAK_SAN_WN_2026_8_23_2050
4772503,2026,8,23,7,WN,1802,MSY,HOU,1625,NaN,...,80.0,72.0,60.0,302.0,1745.0,NaN,NaN,0,0,MSY_HOU_WN_2026_8_23_1625
4772504,2026,8,23,7,WN,695,LAS,ATL,555,NaN,...,245.0,238.0,213.0,1747.0,1300.0,NaN,NaN,0,0,LAS_ATL_WN_2026_8_23_555


In [38]:
logs['timestamp'].min(), logs['timestamp'].max()

('2026-02-25T11:59:55Z', '2026-02-25T20:05:00Z')

In [39]:
logs['timestamp'].value_counts()

timestamp
2026-02-25T12:19:59Z    49
2026-02-25T12:24:58Z    46
2026-02-25T12:54:59Z    45
2026-02-25T12:04:58Z    44
2026-02-25T12:59:58Z    40
                        ..
2026-02-25T14:39:17Z     1
2026-02-25T14:45:00Z     1
2026-02-25T14:54:52Z     1
2026-02-25T15:19:55Z     1
2026-02-25T15:24:56Z     1
Name: count, Length: 649, dtype: int64

In [41]:
logs.columns

Index(['flight', 'lat', 'lon', 'alt', 'gspeed', 'vspeed', 'timestamp',
       'orig_iata', 'dest_iata', 'eta'],
      dtype='str')

In [42]:
flights['FLIGHT_NUMBER'] = flights['AIRLINE'] + flights['FLIGHT_NUMBER'].astype(str)

In [43]:
test = logs.merge(
    flights[['FLIGHT_NUMBER', 'FLIGHT_KEY']],
    left_on=['flight'],
    right_on=['FLIGHT_NUMBER'],
    how='left'
)

In [44]:
test.head()

,flight,lat,lon,alt,gspeed,vspeed,timestamp,orig_iata,dest_iata,eta,FLIGHT_NUMBER,FLIGHT_KEY
0,AA1349,32.81462,-95.80302,40000,346,0,2026-02-25T15:59:57Z,JFK,AUS,2026-02-25T16:34:08Z,AA1349,RDU_DFW_AA_2025_8_29_738
1,AA1349,32.81462,-95.80302,40000,346,0,2026-02-25T15:59:57Z,JFK,AUS,2026-02-25T16:34:08Z,AA1349,MSY_DFW_AA_2025_8_29_635
2,AA1349,32.81462,-95.80302,40000,346,0,2026-02-25T15:59:57Z,JFK,AUS,2026-02-25T16:34:08Z,AA1349,MSY_DFW_AA_2025_8_31_635
3,AA1349,32.81462,-95.80302,40000,346,0,2026-02-25T15:59:57Z,JFK,AUS,2026-02-25T16:34:08Z,AA1349,RDU_DFW_AA_2025_9_1_715
4,AA1349,32.81462,-95.80302,40000,346,0,2026-02-25T15:59:57Z,JFK,AUS,2026-02-25T16:34:08Z,AA1349,MSY_DFW_AA_2025_9_3_635


In [45]:
logs['airline'] = logs['flight'].str[:2]

In [58]:
logs.head()

,flight,lat,lon,alt,gspeed,vspeed,timestamp,orig_iata,dest_iata,eta,airline,date,date_year,date_month,date_day
0,AA1349,32.81462,-95.80302,40000,346,0,2026-02-25 15:59:57+00:00,JFK,AUS,2026-02-25T16:34:08Z,AA,2026-02-25,2026,2,25
1,DL2301,31.69109,-85.77339,39000,502,0,2026-02-25 15:59:58+00:00,MSP,TPA,2026-02-25T16:44:48Z,DL,2026-02-25,2026,2,25
2,DL2582,35.93753,-115.38531,39000,436,0,2026-02-25 15:59:59+00:00,BUR,SLC,2026-02-25T16:59:44Z,DL,2026-02-25,2026,2,25
3,DL8851,37.86596,-100.55422,40000,368,0,2026-02-25 15:59:59+00:00,ATL,DEN,2026-02-25T16:41:04Z,DL,2026-02-25,2026,2,25
4,UA1384,38.17987,-101.32871,38025,386,0,2026-02-25 15:59:58+00:00,ICT,DEN,2026-02-25T16:40:32Z,UA,2026-02-25,2026,2,25


In [57]:
logs['date_year'] = logs['timestamp'].dt.year
logs['date_month'] = logs['timestamp'].dt.month
logs['date_day'] = logs['timestamp'].dt.day   # or .dt.day if you want just the number

In [62]:
flights.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,...,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,FLIGHT_KEY
0,2025,8,29,5,US,US1919,JFK,CLT,1700,1725.0,...,121.0,117.0,85.0,541.0,1901.0,1922.0,21.0,0,0,JFK_CLT_US_2025_8_29_1700
1,2025,8,29,5,US,US2067,CLT,DTW,1755,1752.0,...,107.0,96.0,79.0,500.0,1942.0,1928.0,-14.0,0,0,CLT_DTW_US_2025_8_29_1755
2,2025,8,29,5,B6,B6862,SJU,BOS,1931,2143.0,...,241.0,241.0,225.0,1674.0,2232.0,44.0,132.0,0,0,SJU_BOS_B6_2025_8_29_1931
3,2025,8,29,5,WN,WN683,HOU,TUL,1300,1311.0,...,85.0,85.0,67.0,453.0,1425.0,1436.0,11.0,0,0,HOU_TUL_WN_2025_8_29_1300
4,2025,8,29,5,OO,OO4489,OMA,MSP,1927,1922.0,...,71.0,63.0,51.0,282.0,2038.0,2025.0,-13.0,0,0,OMA_MSP_OO_2025_8_29_1927


In [63]:
test = logs.merge(
    flights[['FLIGHT_NUMBER', 'FLIGHT_KEY', 'YEAR', 'MONTH', 'DAY']],
    left_on=['flight', 'date_year', 'date_month', 'date_day'],
    right_on=['FLIGHT_NUMBER', 'YEAR', 'MONTH', 'DAY'],
    how='left'
)

In [65]:
test.columns

Index(['flight', 'lat', 'lon', 'alt', 'gspeed', 'vspeed', 'timestamp',
       'orig_iata', 'dest_iata', 'eta', 'airline', 'date', 'date_year',
       'date_month', 'date_day', 'FLIGHT_NUMBER', 'FLIGHT_KEY', 'YEAR',
       'MONTH', 'DAY'],
      dtype='str')

In [66]:
test.drop(columns=['airline', 'date', 'date_year', 'date_month', 'date_day', 'FLIGHT_NUMBER', 'YEAR', 'MONTH', 'DAY'], inplace=True)

In [67]:
test.rename(columns={'FLIGHT_KEY': 'flight_key'}, inplace=True)

In [70]:
test.drop_duplicates(inplace=True)

In [71]:
test.shape

(6592, 11)

In [72]:
test.head()

,flight,lat,lon,alt,gspeed,vspeed,timestamp,orig_iata,dest_iata,eta,flight_key
0,AA1349,32.81462,-95.80302,40000,346,0,2026-02-25 15:59:57+00:00,JFK,AUS,2026-02-25T16:34:08Z,JFK_AUS_AA_2026_2_25_726
1,DL2301,31.69109,-85.77339,39000,502,0,2026-02-25 15:59:58+00:00,MSP,TPA,2026-02-25T16:44:48Z,MSP_TPA_DL_2026_2_25_806
2,DL2582,35.93753,-115.38531,39000,436,0,2026-02-25 15:59:59+00:00,BUR,SLC,2026-02-25T16:59:44Z,BUR_SLC_DL_2026_2_25_729
3,DL8851,37.86596,-100.55422,40000,368,0,2026-02-25 15:59:59+00:00,ATL,DEN,2026-02-25T16:41:04Z,ATL_DEN_DL_2026_2_25_836
4,UA1384,38.17987,-101.32871,38025,386,0,2026-02-25 15:59:58+00:00,ICT,DEN,2026-02-25T16:40:32Z,ICT_DEN_UA_2026_2_25_927


In [84]:
test.to_csv('~/SkyGraphProject/dataset/logs.csv', index=False)

In [80]:
import os

In [83]:
current_directory = os.getcwd()
print(current_directory)

FileNotFoundError: [Errno 2] No such file or directory

In [3]:
logs.head()

,flight,lat,lon,alt,gspeed,vspeed,timestamp,orig_iata,dest_iata,eta,flight_key
0,AA1349,32.81462,-95.80302,40000,346,0,2026-02-25 15:59:57+00:00,JFK,AUS,2026-02-25T16:34:08Z,JFK_AUS_AA_2026_2_25_726
1,DL2301,31.69109,-85.77339,39000,502,0,2026-02-25 15:59:58+00:00,MSP,TPA,2026-02-25T16:44:48Z,MSP_TPA_DL_2026_2_25_806
2,DL2582,35.93753,-115.38531,39000,436,0,2026-02-25 15:59:59+00:00,BUR,SLC,2026-02-25T16:59:44Z,BUR_SLC_DL_2026_2_25_729
3,DL8851,37.86596,-100.55422,40000,368,0,2026-02-25 15:59:59+00:00,ATL,DEN,2026-02-25T16:41:04Z,ATL_DEN_DL_2026_2_25_836
4,UA1384,38.17987,-101.32871,38025,386,0,2026-02-25 15:59:58+00:00,ICT,DEN,2026-02-25T16:40:32Z,ICT_DEN_UA_2026_2_25_927


In [5]:
from datetime import datetime

def convert_timestamp(ts_str):
    # Converte la stringa ISO con spazio in un oggetto datetime
    dt = datetime.fromisoformat(ts_str)
    # Formatta l'oggetto nel formato richiesto con 'T' e 'Z'
    return dt.strftime('%Y-%m-%dT%H:%M:%SZ')


In [6]:
# ...existing code...
logs['timestamp'] = logs['timestamp'].apply(convert_timestamp)
# ...existing code...

In [7]:
logs.head()

,flight,lat,lon,alt,gspeed,vspeed,timestamp,orig_iata,dest_iata,eta,flight_key
0,AA1349,32.81462,-95.80302,40000,346,0,2026-02-25T15:59:57Z,JFK,AUS,2026-02-25T16:34:08Z,JFK_AUS_AA_2026_2_25_726
1,DL2301,31.69109,-85.77339,39000,502,0,2026-02-25T15:59:58Z,MSP,TPA,2026-02-25T16:44:48Z,MSP_TPA_DL_2026_2_25_806
2,DL2582,35.93753,-115.38531,39000,436,0,2026-02-25T15:59:59Z,BUR,SLC,2026-02-25T16:59:44Z,BUR_SLC_DL_2026_2_25_729
3,DL8851,37.86596,-100.55422,40000,368,0,2026-02-25T15:59:59Z,ATL,DEN,2026-02-25T16:41:04Z,ATL_DEN_DL_2026_2_25_836
4,UA1384,38.17987,-101.32871,38025,386,0,2026-02-25T15:59:58Z,ICT,DEN,2026-02-25T16:40:32Z,ICT_DEN_UA_2026_2_25_927


In [8]:
logs.to_csv('~/SkyGraphProject/dataset/logs.csv', index=False)

In [10]:
logs.sort_values(by='timestamp')

,flight,lat,lon,alt,gspeed,vspeed,timestamp,orig_iata,dest_iata,eta,flight_key
2677,DL42,8.75692,-153.13181,41000,490,0,2026-02-25T11:59:55Z,SYD,LAX,2026-02-25T17:01:52Z,SYD_LAX_DL_2026_2_25_1433
5072,UA2312,23.53732,-93.48418,37000,472,0,2026-02-25T11:59:55Z,SFO,SJO,2026-02-25T14:11:12Z,SFO_SJO_UA_2026_2_25_818
2672,AA2111,35.22007,-80.94247,0,0,0,2026-02-25T11:59:56Z,CLT,LGA,NaN,CLT_LGA_AA_2026_2_25_743
2671,AA2111,35.22007,-80.94247,0,0,0,2026-02-25T11:59:56Z,CLT,LGA,NaN,LGA_CLT_AA_2026_2_25_1014
2666,KL689,61.86017,-34.99528,36975,473,0,2026-02-25T11:59:56Z,AMS,CUN,2026-02-25T19:22:40Z,AMS_CUN_KL_2026_2_25_906
...,...,...,...,...,...,...,...,...,...,...,...
2649,AA3317,39.98972,-74.80073,11575,368,-1408,2026-02-25T20:04:59Z,PBI,LGA,2026-02-25T20:17:18Z,LGA_PBI_AA_2026_2_25_844
2648,AA1616,3.51156,-80.95797,2100,161,-896,2026-02-25T20:04:59Z,MCI,CLT,2026-02-25T20:06:02Z,MCI_CLT_AA_2026_2_25_1232
2663,DL1368,40.64791,-73.77303,0,19,0,2026-02-25T20:04:59Z,MSY,JFK,2026-02-25T20:02:12Z,JFK_MSY_DL_2026_2_25_813
2642,AA3052,33.73393,-7.80031,35025,498,0,2026-02-25T20:05:00Z,FLL,DCA,2026-02-25T20:56:32Z,FLL_DCA_AA_2026_2_25_1901


In [11]:
logs.columns

Index(['flight', 'lat', 'lon', 'alt', 'gspeed', 'vspeed', 'timestamp',
       'orig_iata', 'dest_iata', 'eta', 'flight_key'],
      dtype='str')

In [12]:
logs['gspeed']-logs['gspeed']

0       0
1       0
2       0
3       0
4       0
       ..
6587    0
6588    0
6589    0
6590    0
6591    0
Name: gspeed, Length: 6592, dtype: int64